# Memecoins: Pump or Dump — Baseline Notebook

Workflow: **Colab notebook + Google Drive + GitHub** (same as Zindi/Opencode sessions).

- Username: `kinuthiawachira` (Telegram linked ✅)
- Runtime: **CPU** (free tier)
- Data: `/content/data/`
- Persistence: `/content/drive/MyDrive/kaggle/memecoins-pump-or-dump/`

Run cells top to bottom. Do **not** copy-paste — this file lives in GitHub and Drive.

## Colab Secrets (key icon 🔑)
Add these two secrets **before** running Cell 2:
- `KAGGLE_USERNAME` — your Kaggle username
- `KAGGLE_API_TOKEN` — your Kaggle API key (kaggle.com → Account → API → Create New API Token)

Access via `from google.colab import userdata; userdata.get('secret_name')`
Enable **Notebook access** on both, then **Runtime → Restart session**.

In [ ]:
# Cell 1 — Mount Google Drive and set up project paths
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DRIVE_PROJECT = Path('/content/drive/MyDrive/kaggle/memecoins-pump-or-dump')
DATA_DIR = Path('/content/data')
OUTPUT_DIR = Path('/content/outputs')

for p in [DRIVE_PROJECT, DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Drive project:', DRIVE_PROJECT)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)
print('Drive contents:', [p.name for p in DRIVE_PROJECT.iterdir()])


In [ ]:
# Cell 2 — Install deps + Kaggle CLI, write kaggle.json from uploaded file
!pip install -q kaggle pandas numpy scikit-learn xgboost lightgbm

import json
from pathlib import Path

# Read kaggle.json from uploaded file in Colab files sidebar
kaggle_json_path = Path('/content/kaggle.json')

if not kaggle_json_path.exists():
    raise ValueError(
        "kaggle.json not found in Colab files sidebar.\n"
        "Upload your Kaggle API token (kaggle.com → Account → API → Create New API Token)\n"
        "to the files sidebar (folder icon 📁), then re-run this cell."
    )

with open(kaggle_json_path) as f:
    creds = json.load(f)

username = creds.get('username')
key = creds.get('key')

# Debug: presence only
print('username found:', bool(username))
print('key found:', bool(key))

if not username or not key:
    raise ValueError(
        "kaggle.json is missing 'username' or 'key' fields.\n"
        "Re-download from kaggle.com → Account → API → Create New API Token."
    )

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump({"username": username, "key": key}, f)

!chmod 600 /root/.kaggle/kaggle.json
!kaggle competitions download -c memecoins-pump-or-dump -p /content/data
!unzip -o /content/data/memecoins-pump-or-dump.zip -d /content/data/

print('Files in /content/data:')
for f in sorted(Path('/content/data').iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')


In [ ]:
# Cell 3 — Explore the data
import pandas as pd
import numpy as np

csv_files = list(DATA_DIR.glob('*.csv'))
print('CSV files:', [f.name for f in csv_files])

for f in csv_files:
    df = pd.read_csv(f, nrows=5)
    print(f'\n=== {f.name} ===')
    print('Shape (first 5 rows):', df.shape)
    print('Columns:', list(df.columns))
    print(df.head())


In [ ]:
# Cell 4 — Load full training data and build baseline
# TODO: update filenames after Cell 3 shows actual names
train_file = None
test_file = None
for f in csv_files:
    name = f.name.lower()
    if 'train' in name:
        train_file = f
    if 'test' in name or 'sample' in name:
        test_file = f

if train_file:
    train = pd.read_csv(train_file)
    print(f'Train shape: {train.shape}')
    print(train.dtypes)
    print(train.describe(include='all'))

if test_file:
    test = pd.read_csv(test_file)
    print(f'\nTest shape: {test.shape}')
    print(test.dtypes)
    print(test.head())


In [ ]:
# Cell 5 — Baseline: XGBoost predicts final_x, outputs tp sell targets
# submission.csv = one tp value per token (0 = do not buy)

import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path

DATA = Path('/content/data')

# ---------- Load core tables ----------
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')

print(f'train: {train.shape}, test: {test.shape}')

# ---------- Token creation features ----------
creation = pd.read_csv(DATA / '03_sample_pumpfun_token_creation.csv')
create_feat = creation.groupby('mint').agg(
    bundle_size=('bundle_size', 'first'),
    gas_used=('gas_used', 'first'),
    dev_balance=('dev_balance', 'first'),
    bundled_buys_count=('bundled_buys_count', 'first'),
    is_cashback_enabled=('is_cashback_enabled', 'first'),
    mayhem_mode=('mayhem_mode', 'first'),
    created_ts=('created_ts', 'first'),
).reset_index()

# ---------- Swap features (first hour after launch) ----------
swaps = pd.read_csv(DATA / '00_sample_pumpfun_swaps.csv')
swaps = swaps.merge(creation[['mint', 'created_ts']], on='mint', how='left')
swaps['ts_rel'] = swaps['ts'] - swaps['created_ts']
early = swaps[swaps['ts_rel'] <= 3600]  # first hour

swap_feat = early.groupby('mint').agg(
    early_tx_count=('ts', 'count'),
    early_buyers=('signing_wallet', 'nunique'),
    early_max_price=('new_price', 'max'),
    early_min_price=('new_price', 'min'),
    early_first_price=('new_price', 'first'),
    early_total_sol=('sol', 'sum'),
    early_buy_count=('direction', lambda s: (s == 'buy').sum()),
).reset_index()
swap_feat['early_buy_ratio'] = swap_feat['early_buy_count'] / swap_feat['early_tx_count']
swap_feat['price_mult_1h'] = swap_feat['early_max_price'] / swap_feat['early_first_price']

# ---------- Merge into train ----------
df = train.merge(create_feat, on='mint', how='left')
df = df.merge(swap_feat, on='mint', how='left')

# ---------- Train XGBoost on final_x ----------
feature_cols = [
    'bundle_size', 'gas_used', 'dev_balance', 'bundled_buys_count',
    'is_cashback_enabled', 'mayhem_mode',
    'early_tx_count', 'early_buyers', 'early_max_price', 'early_min_price',
    'early_first_price', 'early_total_sol', 'early_buy_count',
    'early_buy_ratio', 'price_mult_1h',
]
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    df[c] = df[c].fillna(df[c].median())

X = df[feature_cols]
y = df['final_x']

model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model.fit(X, y)
print('XGBoost trained. Top features:')
imp = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_})
print(imp.sort_values('importance', ascending=False).head(8))

# ---------- Predict on test ----------
test_feat = test.merge(create_feat, on='mint', how='left')
test_feat = test_feat.merge(swap_feat, on='mint', how='left')
for c in feature_cols:
    test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce')
    test_feat[c] = test_feat[c].fillna(df[c].median())

X_test = test_feat[feature_cols]
preds = model.predict(X_test)

# ---------- Build submission: tp = predicted final_x if > threshold, else 0 ----------
threshold = 1.1  # only buy if expected >10% gain
submission = pd.DataFrame({
    'mint': test['mint'],
    'tp': np.where(preds > threshold, np.round(preds, 4), 0),
})

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} rows to {submission_path}')
print(submission.head(10))
print(f'Buy signals: {(submission["tp"] > 0).sum()} / {len(submission)}')
print(f'Buy rate: {(submission["tp"] > 0).mean():.1%}')
buy = submission[submission['tp'] > 0]
if len(buy) > 0:
    print(f'Mean predicted tp (buy signals): {buy["tp"].mean():.4f}')
    print(f'Max predicted tp: {buy["tp"].max():.4f}')


In [ ]:
# Cell 6 — Save notebook and submission to Google Drive (persists across sessions)
import shutil, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Copy submission to Drive
drive_submission = DRIVE_PROJECT / f'submission_{timestamp}.csv'
if OUTPUT_DIR.joinpath('submission.csv').exists():
    shutil.copy(OUTPUT_DIR / 'submission.csv', drive_submission)
    print(f'Saved submission to Drive: {drive_submission}')

# Save a run marker to Drive
marker = DRIVE_PROJECT / f'ran_{timestamp}.txt'
marker.write_text(f'Baseline run at {timestamp}\n')
print(f'Wrote run marker: {marker}')

print('\nDrive project contents:')
for p in sorted(DRIVE_PROJECT.iterdir()):
    print(f'  {p.name}')
